In [1]:
from autocvd import autocvd
autocvd(num_gpus = 1, interval=1)

from functools import partial

# timing and progress bars
from timeit import default_timer as timer
from tqdm import tqdm
import pickle

# numerics
import jax
import jax.numpy as jnp
import optimistix as optx
import equinox as eqx
import flax
import optax

from jax import jit

from flow import Flow
from dataloader import OdisseoOTAllParametersPosition_newprior


In [2]:
dataloader_train = OdisseoOTAllParametersPosition_newprior(
            DATA_ROOT = '/export/data/vgiusepp/odisseo_data/data_varying_position_newprior/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_newprior',
            num_samples= 100_000,
            split= [0, 94999],
            seed= 0,
            num_steps = 2000,
            batch_size = 32,
            use_jax = False)

SEED = 5678
dim_flow = 13
N_dim = 256
N_head = 2
depth = 2
key = jax.random.PRNGKey(SEED)
key, subkey = jax.random.split(key, 2)
flow = Flow(dim_flow=dim_flow, N_dim=N_dim, N_head=N_head, depth=depth, sample_size=1000)

flow_params = flow.init_weights(subkey)

from flax.training.checkpoints import restore_checkpoint
restore_object = restore_checkpoint('/export/home/vgiusepp/sbi_diff_sim/sbi-sim/flow_matching_personal/checkpoint_0', target=None)

key = jax.random.PRNGKey(0)
t = jnp.linspace(0, 1, 1).reshape(-1, 1)
theta = jax.random.normal(key, (1, 13))
x_obs = jax.random.normal(key, (1, 1000, 6))
flow.apply({'params': restore_object}, timesteps=t, sample=theta, encoder_hidden_states=x_obs)

odisseo_AllParametersPosition_newprior
DATA_ROOT: /export/data/vgiusepp/odisseo_data/data_varying_position_newprior/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_newprior/
Loading odisseo_AllParametersPosition_newprior locally from existing directory /export/data/vgiusepp/odisseo_data/data_varying_position_newprior/sbi_sim/data/sbi-benchmarks/odisseo_AllParametersPosition_newprior
using numpy to load
using numpy to load
Dataloader sampling without replacements (num_steps: 2969)
compiling.....
normalizing...
x shape (1000, 6)
encoder_hidden_states after norm (1, 1000, 6)
timesteps (1,)
t_emb (1, 1024)
x shape in MAB (1, 1000, 256)
x shape in MAB (1, 1000, 256)
x shape in MAB (1, 1, 256)


/export/home/vgiusepp/miniconda3/envs/sbi_sim/lib/python3.12/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1251: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


compiling.....
normalizing...
x shape (1000, 6)
encoder_hidden_states after norm (1, 1000, 6)
timesteps (1, 1)
t_emb (1, 1024)
x shape in MAB (1, 1000, 256)
x shape in MAB (1, 1000, 256)
x shape in MAB (1, 1, 256)


Array([[ 3.6911833 , -9.545572  ,  7.3389688 , 15.1910305 , -6.922135  ,
         0.88468903,  3.3849351 ,  5.530689  ,  0.84419227,  0.13850453,
         5.1635094 ,  9.199073  ,  7.2010174 ]], dtype=float32)